# Exemples de mise en oeuvre des outils d'optimisation de la libriarie `scipy`



In [42]:
%load_ext autoreload
%autoreload 2

In [ ]:
import scipy.optimize as opt
import numpy as np
import scipy.optimize as opt
import pickle
from scipy.ndimage.filters import gaussian_filter
import matplotlib.pyplot as plt
import numpy as np
from scipy import interpolate
from matplotlib import rc
rc('text', usetex=True)
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)


## Problème considéré


## Déclaration de la fonction objectif : Elevation

In [ ]:
#filename = 'results_parametric_tet4.pck'
filename = 'results_parametric_tet4_f1o15Hz_h37_25x25points.pck'
f=open(filename,'rb')
frf_tet4_xfem = pickle.load(f)
f.close()

nbval = frf_tet4_xfem[0].shape[0]
X_0 = frf_tet4_xfem[0][0]
X = frf_tet4_xfem[0].reshape((nbval, nbval))
Y_0 = frf_tet4_xfem[0][0]
Y = frf_tet4_xfem[1].reshape((nbval, nbval))
val_p_0 = frf_tet4_xfem[2]
val_p = frf_tet4_xfem[2].reshape((nbval, nbval))/(1000*9.81)
val_f = frf_tet4_xfem[3].reshape((nbval, nbval))

val_f_max = 19
val_f_ref=val_f
val_f=-(val_f-val_f_max)

sigma=0.9
val_p = gaussian_filter(val_p, sigma)
sigma=1.2
val_f = gaussian_filter(val_f, sigma)
val_f_ref = gaussian_filter(val_f_ref, sigma)

#X : lup
#Y : ldown
print(X_0)
print(Y_0)
print(val_p_0)

In [ ]:
X_0.shape

In [ ]:
Y_0

In [ ]:
# funObj= lambda x: 100*(x[1]-x[0]**2)**2 + (1-x[0])**2
# funGradObj=lambda x: np.array([-400*x[0]*(x[1]-x[0]**2)-2*(1-x[0]),200*(x[1]-x[0]**2)])

In [ ]:
val_p

In [ ]:
#interp_val_p = RegularGridInterpolator((X_0, Y_0), val_p)
#interp_val_f = RegularGridInterpolator((X_0, Y_0), val_f)

interp_val_p_tmp = interpolate.RegularGridInterpolator(
    (X_0, Y_0), val_p.T, method="linear"
)

def interp_val_p(point):
    return interp_val_p_tmp(point)#[0],point[1])
       
interp_val_f_tmp = interpolate.RegularGridInterpolator(
    (X_0, Y_0), val_f.T, method="linear"
) 

def interp_val_f(point):
    return interp_val_f_tmp(point)#[0],point[1])

interp_val_f_ref_tmp = interpolate.RegularGridInterpolator(
    (X_0, Y_0), val_f_ref.T, method="linear"
) 

def interp_val_f_ref(point):
    return interp_val_f_ref_tmp(point)#[0],point[1])

# test :
point = np.array([0, -0.2])
#point=[-2.000e-01 , -1.181e-01]
print(interp_val_p(point))
print(interp_val_f(point))

In [ ]:
print(val_p[0][0])
print(X_0[0])
print(Y_0[0])
print(val_p[nbval-1][nbval-1])
print(X_0[nbval-1])
print(Y_0[nbval-1])

In [ ]:
points = np.array([[X_0[0], Y_0[0]],[X_0[nbval-1], Y_0[nbval-1]]])
interp_val_p(points)

In [ ]:
val_p_check = interp_val_p(np.vstack((X.flatten(), Y.flatten())).T).reshape(X.shape)
val_f_ref_check = interp_val_f_ref(np.vstack((X.flatten(), Y.flatten())).T).reshape(X.shape)
val_f_check = interp_val_f(np.vstack((X.flatten(), Y.flatten())).T).reshape(X.shape)

In [ ]:
plt.contour(X,Y,val_p*1e3,20)
plt.xlabel('x_up')
plt.ylabel('x_down')
plt.title('Elevation')

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(X,Y,val_p*1e3, cmap='inferno', linewidth=0, antialiased=False)
ax.scatter(X,Y,val_p*1e3, color='k', s=1)
ax.scatter(X,Y,val_p_check*1e3, color='k', s=1)
#plt.plot(-2.000e-01,-1.181e-01,0.00208525*1e3,'ok')
#point=np.array([-2.000e-01 , -1.181e-01])
#point=np.array([0 , -0.2])
fig.colorbar(surface, shrink=0.7, aspect=10)

#point=np.array([-1.181e-01, -2.000e-01])
#plt.plot(point[0],point[1],interp_val_p(point)*1e3,'ob')

ax.set_xlabel('$x_{up}$ [m]')
ax.set_ylabel('$x_{down}$ [m]')
ax.set_zlabel('Elevation [mm]')
ax.set_title('Point A elevation [mm]')
plt.savefig('img_elevation_response_surface_2_param_aachen.png', dpi=200)  
#plt.savefig('img_elevation_response_surface_2_param_aachen_optim_point.png', dpi=200)  
plt.show()

In [ ]:
interp_val_p([0,-0.2])*1e3

## Déclaration des contraintes : Force

In [ ]:
plt.contour(X,Y,val_f,20)
plt.xlabel('x_up')
plt.ylabel('x_down')
plt.title('Force')

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(X,Y,val_f_ref, cmap='inferno', linewidth=0, antialiased=False)


fig.colorbar(surface, shrink=0.7, aspect=10)

#point=np.array([-1.181e-01, -2.000e-01])
#plt.plot(point[0],point[1],interp_val_f(point)+val_f_max,'ob')

ax.set_xlabel('$x_{up}$ [m]')
ax.set_ylabel('$x_{down}$ [m]')

ax.set_zlabel('Force on structure [N]')
ax.set_title('Force on structure [N]')
plt.savefig('img_force_response_surface_2_param_aachen.png', dpi=200)  
#plt.savefig('img_force_response_surface_2_param_aachen_optim_point.png', dpi=200)  

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
surface = ax.plot_surface(X,Y,val_f, cmap='inferno', linewidth=0, antialiased=False)
ax.scatter(X,Y,val_f, color='k', s=1)
ax.scatter(X,Y,val_f_check, color='k', s=1)
fig.colorbar(surface, shrink=0.7, aspect=10)

#point=np.array([-1.181e-01, -2.000e-01])
#plt.plot(point[0],point[1],interp_val_f(point)+val_f_max,'ob')

ax.set_xlabel('$x_{up}$ [m]')
ax.set_ylabel('$x_{down}$ [m]')
ax.set_zlabel('Constraint function [N]')
ax.set_title('Constraint function [N]')
# plt.savefig('img_force_response_surface_2_param_aachen.png', dpi=200)  
#plt.savefig('img_force_response_surface_2_param_aachen_optim_point.png', dpi=200)  


plt.show()

### Sans les gradients

In [ ]:
#ineq_cons_wo_grad = {'type': 'ineq',
#             'fun' : lambda x: np.array([1 - x[0] - 2*x[1],
#                                         1 - x[0]**2 - x[1],
#                                         1 - x[0]**2 + x[1]])}

#eq_cons_wo_grad = {'type': 'eq',
#           'fun' : lambda x: np.array([2*x[0] + x[1] - 1])}

ineq_cons_wo_grad = {'type': 'ineq',
             'fun' : lambda x: interp_val_f(x)}



## Execution de l'optimisation

### Décalaration des bornes des paramètres

In [ ]:
# bounds = opt.Bounds([0, -0.5], [1.0, 2.0])
bounds_opt = [(-0.2,0.2),(-0.2,0.2)]

### Execution optimisation sans gradients

In [ ]:
## 
x0 = np.array([0.0, 0.0])
res = opt.minimize(interp_val_p,
                x0, 
               method='SLSQP', # 
               jac=None,
               constraints=[ineq_cons_wo_grad], 
               options={'ftol': 1e-9, 'disp': True},
               bounds=bounds_opt)
print(res)

In [ ]:
# test :
point = np.array([-2.000e-01, -1.181e-01])
point = np.array([-2.000e-01, -2.000e-01])
print(interp_val_p(point))
print(interp_val_f(point))

### Extraction des informations au long des iterations

#### Class for storage

In [ ]:
class objStore:
    def __init__(self):
        self.x=[]
        self.fun=[]
    def store(self,x,f):
        self.x.append(x)
        self.fun.append(f)
data = objStore()

### modification of the objective function to store the data

In [ ]:
def funObjMod(x):
    f = interp_val_p(x)
    data.store(x,f)
    return f

### Run optimization without constraint

In [ ]:
data_wo_cons = objStore()
def test_call_wo_cons(xk):
    data_wo_cons.store(xk, interp_val_p(xk))

In [ ]:
x0 = np.array([0, 0])
res = opt.minimize(funObjMod,
                x0, 
               method='SLSQP', 
               jac=None,
               options={'ftol': 1e-9, 'disp': True},
               callback=test_call_wo_cons,
               bounds=bounds_opt)
print(res)

In [ ]:
print(data_wo_cons.x)

In [ ]:
plt.step(range(len(data_wo_cons.x)),data_wo_cons.fun,where='post')
plt.xlabel('Iteration')
plt.ylabel('Objective value')

In [ ]:
# plt.rcParams.update({'usetex': True})
plt.contour(X,Y,val_p*1e3,20)
xd = [v[1] for v in data_wo_cons.x]
xu = [v[0] for v in data_wo_cons.x]
shift = 5e-3
for i in range(0, len(xd)):
    plt.plot(xu[i:i+2], xd[i:i+2], 'r-', linewidth=0.5)
    it = i+1
    plt.text(xu[i]-shift, xd[i]+shift, f'{it}', fontsize=10, ha='right', va='bottom')
plt.plot(xu[-1], xd[-1], '*', color='blue', markersize=15)
plt.scatter(xu, xd, color='red', label='Optimized Point')
plt.xlabel('$x_{up}$')#, usetex=True)
plt.ylabel('$x_{down}$')
plt.title('Elevation')

### Run optimization with constraint

In [ ]:
data_w_cons = objStore()
def test_call_w_cons(xk):
    data_w_cons.store(xk, interp_val_p(xk))

In [ ]:
x0 = np.array([0, 0])
res = opt.minimize(funObjMod,
                x0, 
               method='SLSQP', 
               jac=None,
               constraints=[ineq_cons_wo_grad], 
               options={'ftol': 1e-9, 'disp': True},
               callback=test_call_w_cons,
               bounds=bounds_opt)
print(res)

In [ ]:
print(data_w_cons.x)

In [ ]:
plt.step(range(len(data_w_cons.x)),data_w_cons.fun,where='post')
plt.xlabel('Iteration')
plt.ylabel('Objective value')

In [ ]:
# plt.rcParams.update({'usetex': True})
plt.contour(X,Y,val_p*1e3,20)
xd = [v[1] for v in data_w_cons.x]
xu = [v[0] for v in data_w_cons.x]
shift = 5e-3
for i in range(0, len(xd)):
    plt.plot(xu[i:i+2], xd[i:i+2], 'r-', linewidth=0.5)
    it = i+1
    plt.text(xu[i]-shift, xd[i]+shift, f'{it}', fontsize=10, ha='right', va='bottom')
plt.plot(xu[-1], xd[-1], '*', color='blue', markersize=15)
plt.scatter(xu, xd, color='red', label='Optimized Point')
plt.xlabel('$x_{up}$')#, usetex=True)
plt.ylabel('$x_{down}$')
plt.title('Elevation')

In [ ]:
plt.contour(X,Y,val_p,20)
xd = [v[1] for v in data_w_cons.x]
xu = [v[0] for v in data_w_cons.x]
shift = 5e-3
for i in range(0, len(xd)):
    plt.plot(xu[i:i+2], xd[i:i+2], 'r-', linewidth=0.5)
    it = i+1
    plt.text(xu[i]-shift, xd[i]+shift, f'{it}', fontsize=12, ha='right', va='bottom')
plt.plot(xu[-1], xd[-1], '*', color='blue', markersize=15)
plt.scatter(xu, xd, color='red', label='Optimized Point')
plt.xlabel('$x_{up}$')# usetex=True)
plt.ylabel('$x_{down}$')
plt.title('Elevation')

# Mise en oeuvre CBO

In [ ]:
# load packages
import botorch,gpytorch,torch
from loguru import logger
import numpy as np
import matplotlib.pyplot as plt
import bo_lib

In [ ]:
# data definition
default_values = bo_lib.default_values()
nb_samples = 5
bounds = [[-0.2, -0.2], [0.2, 0.2]]
#
def init_GP_structure():
    likelihood_obj = gpytorch.likelihoods.FixedNoiseGaussianLikelihood
    mean_obj = gpytorch.means.LinearMean #gpytorch.means.ConstantMean
    covar_obj = gpytorch.kernels.MaternKernel
    GP_type_obj = botorch.models.SingleTaskGP
    mll_obj = gpytorch.mlls.ExactMarginalLogLikelihood
    optim_obj = torch.optim.Adam
    return likelihood_obj, mean_obj, covar_obj, GP_type_obj, mll_obj, optim_obj
#
noise_lvl = 1e-4
nb_it_optim = 100
learning_rate=0.1
nb_it_bo = 20



In [ ]:
# data for plotting
nb_samples_plot = 100
Xlin = np.linspace(bounds[0][0], bounds[1][0], nb_samples_plot)
Ylin = np.linspace(bounds[0][1], bounds[1][1], nb_samples_plot+1)
X_plot,Y_plot = np.meshgrid(Xlin, Ylin)
XYeval = np.vstack((X_plot.flatten(), Y_plot.flatten())).T
Z_plot = interp_val_p(XYeval)
Z_plot = Z_plot.reshape(X_plot.shape)
f_plot = interp_val_f_ref(XYeval)
f_plot = f_plot.reshape(X_plot.shape)
C_plot = interp_val_f(XYeval)
C_plot = C_plot.reshape(X_plot.shape)
#
fig = plt.figure()
ax1=plt.subplot(231,projection='3d')
s = ax1.plot_surface(X_plot,Y_plot,Z_plot*1e3,cmap=plt.cm.viridis)
fig.colorbar(s, shrink=0.7, aspect=10)
# ax1.plot_surface(X_plot, Y_plot, Z_plot, alpha=0.5,zorder=100)
ax1.set_xlabel('$x_{up}$')
ax1.set_ylabel('$x_{down}$')
# ax1.set_zlim(-8,10)
ax1.set_zlabel('Objective [mm]')
#
ax1=plt.subplot(234)
s = ax1.contour(X_plot,Y_plot,Z_plot*1e3,cmap=plt.cm.viridis)
ax1.set_xlabel('$x_{up}$')
ax1.set_ylabel('$x_{down}$')
ax1.contour(X_plot, Y_plot, C_plot, [0], colors='red')
plt.contourf(X_plot, Y_plot, C_plot, levels=[-10, 0], colors=["red"], alpha=0.2)
#
ax1=plt.subplot(232,projection='3d')
s = ax1.plot_surface(X_plot, Y_plot, f_plot, cmap=plt.cm.viridis)
ax1.contour(X_plot, Y_plot, f_plot, [val_f_max]) 

fig.colorbar(s, shrink=0.7, aspect=10)
ax1.set_xlabel('$x_{up}$')
ax1.set_ylabel('$x_{down}$')
# ax1.set_zlim(-8,10)
ax1.set_zlabel('Force [N]')
#
ax1=plt.subplot(235)
s = ax1.contour(X_plot,Y_plot,f_plot,cmap=plt.cm.viridis)
ax1.contour(X_plot, Y_plot, f_plot, [val_f_max], colors='red')
plt.contourf(X_plot, Y_plot, f_plot, levels=[val_f_max, 50], colors=["red"], alpha=0.2)
ax1.set_xlabel('$x_{up}$')
ax1.set_ylabel('$x_{down}$')
#
ax1=plt.subplot(233,projection='3d')
s = ax1.plot_surface(X_plot, Y_plot, C_plot, cmap=plt.cm.viridis)
ax1.contour(X_plot, Y_plot, C_plot, [0]) 
fig.colorbar(s, shrink=0.7, aspect=10)
ax1.set_xlabel('$x_{up}$')
ax1.set_ylabel('$x_{down}$')
# ax1.set_zlim(-8,10)
ax1.set_zlabel('Constraint')
#
ax1=plt.subplot(236)
s = ax1.contour(X_plot,Y_plot,C_plot,cmap=plt.cm.viridis)
ax1.contour(X_plot, Y_plot, C_plot, [0], colors='red')
plt.contourf(X_plot, Y_plot, C_plot, levels=[-10, 0], colors=["red"], alpha=0.2)
ax1.set_xlabel('$x_{up}$')
ax1.set_ylabel('$x_{down}$')
fig.tight_layout()

In [ ]:
XYeval.shape

In [ ]:
bounds[0]

In [ ]:
# sampling
samples = bo_lib.lhs_distrib(bounds, nbs=nb_samples)
Zsamples = interp_val_p(samples)
Zsamples = torch.tensor(Zsamples[np.newaxis].T, dtype=torch.float64)
Csamples = interp_val_f(samples)
Csamples = torch.tensor(Csamples[np.newaxis].T, dtype=torch.float64)


In [ ]:
dataStdizeX = bo_lib.stdize(samples)
dataStdizeZ = bo_lib.stdize(Zsamples)
dataStdizeC = bo_lib.stdize(Csamples)
Xstdize = dataStdizeX.output_data
Zstdize = dataStdizeZ.output_data
Cstdize = dataStdizeC.output_data
Cstdize = [Cstdize]

In [ ]:
gp_train_Z = bo_lib.init_surrogate_model(Xstdize, Zstdize)
gp_train_C = [bo_lib.init_surrogate_model(Xstdize, Cstdize[0])]
# build multi independent GP
multiZC = torch.hstack((Zstdize, *Cstdize))
gp_train_multi = bo_lib.init_surrogate_model(Xstdize, multiZC)

# check interpolation
Z_GP_init_at_samples = gp_train_Z(Xstdize).mean.detach()
logger.info(f' Z maxi interpolation error {(Zstdize.flatten()-Z_GP_init_at_samples).abs().max()}')
for itcons,consI in enumerate(gp_train_C):
    C_GP_init_at_samples = consI(Xstdize).mean.detach()
    logger.info(f'{itcons} C maxi interpolation error {(Cstdize[0].flatten()-C_GP_init_at_samples).abs().max()}')
multi_GP_init_at_samples = gp_train_multi(Xstdize).mean.detach()
for itGP,gpI in enumerate(multi_GP_init_at_samples):
    logger.info(f'Index {itGP} maxi interpolation error {(multiZC[:,itGP].flatten()-gpI).abs().max()}')

In [ ]:
# # partial use of samples
# nbsamples_used=5
# gp_train_Z_test = init_surrogate_model(Xtest[:nbsamples_used,:], 
#                                   Ztest[:nbsamples_used].unsqueeze(-1))
# Z_GP_at_samples = gp_train_Z_test(Xtest[:nbsamples_used,:]).mean.detach()
# logger.info(Ztest[:nbsamples_used].double().flatten()-Z_GP_at_samples)

In [ ]:
# plot GP
XYeval_stdize = dataStdizeX.stdize(torch.tensor(XYeval, dtype=torch.float64))
predictor_Z = gp_train_Z(XYeval_stdize)
Z_GP = predictor_Z.mean.detach()
Z_GP_unstdize = dataStdizeZ.unstdize(Z_GP)
Z_GP_plot = Z_GP_unstdize.reshape(X_plot.shape)

fig = plt.figure()
ax1=plt.subplot(231,projection='3d')
s = ax1.plot_surface(X_plot,Y_plot,Z_GP_plot*1e3,cmap=plt.cm.viridis)
ax1.scatter(samples[:,0].numpy(),samples[:,1].numpy(),Zsamples.numpy().flatten()*1e3,color='red',s=50)
# fig.colorbar(s, shrink=0.7, aspect=10)
# ax1.plot_surface(X_plot, Y_plot, Z_plot, alpha=0.5,zorder=100)
ax1.set_xlabel('$x_{up}$')
ax1.set_ylabel('$x_{down}$')
# ax1.set_zlim(-8,10)
ax1.set_title('Objective (GP) [mm]')


predictor_C = gp_train_C[0](XYeval_stdize)
C_GP = predictor_C.mean.detach()
C_GP_unstdize = dataStdizeC.unstdize(C_GP)
C_GP_plot = C_GP_unstdize.reshape(X_plot.shape)

fig = plt.figure()
ax1=plt.subplot(231,projection='3d')
s = ax1.plot_surface(X_plot,Y_plot,C_GP_plot,cmap=plt.cm.viridis)
ax1.scatter(samples[:,0].numpy(),samples[:,1].numpy(),Csamples.numpy().flatten(),color='red',s=50)
# fig.colorbar(s, shrink=0.7, aspect=10)
# ax1.plot_surface(X_plot, Y_plot, Z_plot, alpha=0.5,zorder=100)
ax1.set_xlabel('$x_{up}$')
ax1.set_ylabel('$x_{down}$')
# ax1.set_zlim(-8,10)
ax1.set_title('Constraint (GP) [N]')

In [ ]:
# plot GP via multi GP
XYeval_stdize = dataStdizeX.stdize(torch.tensor(XYeval, dtype=torch.float64))
predictor_multi = gp_train_multi(XYeval_stdize)
Z_GP_multi = predictor_multi[0].mean.detach()
Z_GP_multi_unstdize = dataStdizeZ.unstdize(Z_GP_multi)
Z_GP_multi_plot = Z_GP_multi_unstdize.reshape(X_plot.shape)

fig = plt.figure()
ax1=plt.subplot(231,projection='3d')
s = ax1.plot_surface(X_plot,Y_plot,Z_GP_multi_plot*1e3,cmap=plt.cm.viridis)
ax1.scatter(samples[:,0].numpy(),samples[:,1].numpy(),Zsamples.numpy().flatten()*1e3,color='red',s=50)
# fig.colorbar(s, shrink=0.7, aspect=10)
# ax1.plot_surface(X_plot, Y_plot, Z_plot, alpha=0.5,zorder=100)
ax1.set_xlabel('$x_{up}$')
ax1.set_ylabel('$x_{down}$')
# ax1.set_zlim(-8,10)
ax1.set_title('Objective (GP) [mm]')


C_GP_multi = predictor_multi[1].mean.detach()
C_GP_multi_unstdize = dataStdizeC.unstdize(C_GP_multi)
C_GP_multi_plot = C_GP_multi_unstdize.reshape(X_plot.shape)

fig = plt.figure()
ax1=plt.subplot(231,projection='3d')
s = ax1.plot_surface(X_plot,Y_plot,C_GP_multi_plot,cmap=plt.cm.viridis)
ax1.scatter(samples[:,0].numpy(),samples[:,1].numpy(),Csamples.numpy().flatten(),color='red',s=50)
# fig.colorbar(s, shrink=0.7, aspect=10)
# ax1.plot_surface(X_plot, Y_plot, Z_plot, alpha=0.5,zorder=100)
ax1.set_xlabel('$x_{up}$')
ax1.set_ylabel('$x_{down}$')
# ax1.set_zlim(-8,10)
ax1.set_title('Constraint (GP) [N]')

### run BO

In [ ]:
XYeval_stdize = dataStdizeX.stdize(torch.tensor(XYeval, dtype=torch.float64))
normalization_data={
    'X': dataStdizeX,
    'Z': dataStdizeZ
}
# run BO
(
    X_update, 
    Z_update, 
    gp_final, 
    GP_at_grid_points, 
    acq_at_grid_points
    ) = bo_lib.execute_CBO(gp_train_Z,
                    fun_obj=interp_val_p,
                    fun_cons=None,
                    bounds=bounds,
                    constraints_bounds=None,
                    nb_it_bo=10, 
                    XY_plot=XYeval_stdize,
                    normalization_data=normalization_data)
    
# gest final best candidate
Xunstdize_final = dataStdizeX.unstdize(X_update)
Zunstdize_final = dataStdizeZ.unstdize(Z_update)
final_best_candidate,IX_final_best,X_final_best = bo_lib.find_best_candidate(Xunstdize_final, Zunstdize_final, constraints_bounds=None)
logger.info('Optimum found: {} at point {}'.format(final_best_candidate.tolist(),X_final_best.tolist()))

In [ ]:
Zunstdize_final

In [ ]:
# plot acquition function/GP function
nb_it_plot = 2
dataACQ = acq_at_grid_points[nb_it_plot]
dataACQgrid = dataACQ.reshape(X_plot.shape)
dataACQgrid[dataACQgrid<1e-8]=1e-8
IXmax = torch.argmax(dataACQgrid.flatten())
logger.info('Max at {} {} (val: {})'.format(X_plot.flatten()[IXmax],Y_plot.flatten()[IXmax],dataACQgrid.flatten()[IXmax]))
dataGP = GP_at_grid_points[nb_it_plot]
dataGPgridtmp = dataStdizeZ.unstdize(dataGP)
dataGPgrid = dataGPgridtmp.reshape(X_plot.shape)
#
fig = plt.figure()
ax1=plt.subplot(231)#,projection='3d')
s = ax1.contourf(X_plot,Y_plot,np.log(dataACQgrid),cmap=plt.cm.viridis,levels=100)
ax1.contour(X_plot,Y_plot,np.log(dataACQgrid),colors='black',linestyles='solid',linewidths=1,levels=20)
# ax1.scatter(Xunstdize_final[:,0],Xunstdize_final[:,1],Zunstdize_final.numpy().flatten()*1e3,color='red',s=20)
# ax1.set_zscale('log', base=2)
fig.colorbar(s, shrink=0.7, aspect=10)
# ax1.plot_surface(X_plot, Y_plot, Z_plot, alpha=0.5,zorder=100)
ax1.set_xlabel('$x_{up}$')
ax1.set_ylabel('$x_{down}$')
# ax1.set_xlim(-0.2,-0.198)
# ax1.set_ylim(-0.2,-0.198)
# ax1.set_zlim(-8,10)
ax1.set_title('Acquisition function')

fig = plt.figure()
ax1=plt.subplot(231,projection='3d')
s = ax1.plot_surface(X_plot,Y_plot,dataGPgrid*1e3,cmap=plt.cm.viridis)
ax1.scatter(Xunstdize_final[:,0],Xunstdize_final[:,1],Zunstdize_final.numpy().flatten()*1e3,color='red',s=20)
# ax1.set_zscale('log', base=2)
# fig.colorbar(s, shrink=0.7, aspect=10)
# ax1.plot_surface(X_plot, Y_plot, Z_plot, alpha=0.5,zorder=100)
ax1.set_xlabel('$x_{up}$')
ax1.set_ylabel('$x_{down}$')
# ax1.set_zlim(-8,10)
ax1.set_title(f'GP at iteration {nb_it_plot+1}')

In [ ]:
# partial use of samples/check interpolation
nbsamples_used=8
newdataStdX = bo_lib.stdize(X_update[:nbsamples_used,:])
Xtest = newdataStdX.output_data
newdataStdZ = bo_lib.stdize(Z_update[:nbsamples_used].unsqueeze(-1))
Ztest = newdataStdZ.output_data
gp_train_Z = bo_lib.init_surrogate_model(Xtest,Ztest)
Z_GP_at_samples = gp_train_Z(Xtest).mean.detach()
logger.info(Ztest.double().flatten()-Z_GP_at_samples)

In [ ]:
# plot GP
XYeval_stdize = dataStdizeX.stdize(torch.tensor(XYeval, dtype=torch.float64))
predictor_Z_final = gp_final(XYeval_stdize)
Z_GP_final = predictor_Z_final.mean.detach()
Z_GP_unstdize_final = dataStdizeZ.unstdize(Z_GP_final)
Z_GP_plot_final = Z_GP_unstdize_final.reshape(X_plot.shape)

fig = plt.figure()
ax1=plt.subplot(231,projection='3d')
s = ax1.plot_surface(X_plot,Y_plot,Z_GP_plot_final*1e3,cmap=plt.cm.viridis)
ax1.scatter(Xunstdize_final[:,0],Xunstdize_final[:,1],Zunstdize_final.numpy().flatten()*1e3,color='red',s=20)
# fig.colorbar(s, shrink=0.7, aspect=10)
# ax1.plot_surface(X_plot, Y_plot, Z_plot, alpha=0.5,zorder=100)
ax1.set_xlabel('$x_{up}$')
ax1.set_ylabel('$x_{down}$')
# ax1.set_zlim(-8,10)
ax1.set_title('Objective (GP) [mm]')


predictor_C = gp_train_C[0](XYeval_stdize)
C_GP = predictor_C.mean.detach()
C_GP_unstdize = dataStdizeC.unstdize(C_GP)
C_GP_plot = C_GP_unstdize.reshape(X_plot.shape)

fig = plt.figure()
ax1=plt.subplot(231,projection='3d')
s = ax1.plot_surface(X_plot,Y_plot,C_GP_plot,cmap=plt.cm.viridis)
# fig.colorbar(s, shrink=0.7, aspect=10)
# ax1.plot_surface(X_plot, Y_plot, Z_plot, alpha=0.5,zorder=100)
ax1.set_xlabel('$x_{up}$')
ax1.set_ylabel('$x_{down}$')
# ax1.set_zlim(-8,10)
ax1.set_title('Constraint (GP) [N]')

### run CBO

In [ ]:
XYeval_stdize = dataStdizeX.stdize(torch.tensor(XYeval, dtype=torch.float64))
constraints_bounds = [(0,torch.inf)]
constraints_bounds_stdize = [dataStdizeC.stdize(torch.tensor(bnd, dtype=torch.float64)) for bnd in constraints_bounds]
normalization_data={
    'X': dataStdizeX,
    'Z': dataStdizeZ,
    'C': dataStdizeC
}
# run BO
(
    CBO_X_update, 
    CBO_ZC_update, 
    CBO_gp_obj_final,     
    CBO_GP_obj_at_grid_points, 
    CBO_GP_cons_at_grid_points,
    CBO_acq_at_grid_points,
    acq_values_final
    ) = bo_lib.execute_CBO(gp_train_multi,
                    fun_obj=interp_val_p,
                    fun_cons=[interp_val_f],
                    bounds=bounds,
                    constraints_bounds=constraints_bounds_stdize,
                    nb_it_bo=50, 
                    XY_plot=XYeval_stdize,
                    normalization_data=normalization_data)
    


In [ ]:
# gest final best candidate
CBO_Xunstdize_final = dataStdizeX.unstdize(CBO_X_update)
CBO_Zunstdize_final = dataStdizeZ.unstdize(CBO_ZC_update[:,0])
CBO_Cunstdize_final = dataStdizeC.unstdize(CBO_ZC_update[:,1:])
(
    final_best_candidate,
    IX_final_best,
    X_final_best
    ) = bo_lib.find_best_candidate(CBO_Xunstdize_final, 
                                   CBO_Zunstdize_final, 
                                   Cval = CBO_Cunstdize_final.T,
                                   constraints_bounds=constraints_bounds)
logger.info('Optimum found: {} at point {}'.format(final_best_candidate.tolist(),X_final_best.tolist()))

In [53]:
# plot acquition function/GP function
for i in range(len(CBO_acq_at_grid_points)):
    nb_it_plot = i
    dataACQ = CBO_acq_at_grid_points[nb_it_plot]
    dataACQgrid = dataACQ.reshape(X_plot.shape)
    dataACQgrid[dataACQgrid<1e-8]=1e-8
    IXmax = torch.argmax(dataACQgrid.flatten())
    logger.info('Max at {} {} (val: {})'.format(X_plot.flatten()[IXmax],Y_plot.flatten()[IXmax],dataACQgrid.flatten()[IXmax]))
    dataGP_obj = CBO_GP_obj_at_grid_points[nb_it_plot]
    dataGPObjgridtmp = dataStdizeZ.unstdize(dataGP_obj)
    dataGPObjgrid = dataGPObjgridtmp.reshape(X_plot.shape)
    dataGP_cons = CBO_GP_cons_at_grid_points[0][nb_it_plot]
    dataGPConsgridtmp = dataStdizeC.unstdize(dataGP_cons)
    dataGPConsgrid = dataGPConsgridtmp.reshape(X_plot.shape)
    nb_points_plot = nb_samples+nb_it_plot
    (
        current_best_candidate,
        IX_current_best,
        X_current_best
        ) = bo_lib.find_best_candidate(CBO_Xunstdize_final[0:nb_points_plot,:], 
                                    CBO_Zunstdize_final[0:nb_points_plot], 
                                    Cval = CBO_Cunstdize_final[0:nb_points_plot].T,
                                    constraints_bounds=constraints_bounds)

    #####################################################################################################
    ######################################################################################################
    ######################################################################################################
    ######################################################################################################
    fig = plt.figure()
    ax1=plt.subplot(111)#,projection='3d')
    s = ax1.contourf(X_plot,Y_plot,np.log(dataACQgrid),cmap=plt.cm.viridis,levels=100)
    ax1.contour(X_plot,Y_plot,np.log(dataACQgrid),colors='black',linestyles='solid',linewidths=1,levels=20)
    ax1.scatter(CBO_Xunstdize_final[0:nb_samples,0],CBO_Xunstdize_final[0:nb_samples,1],color='blue',zorder=1e4)
    ax1.scatter(CBO_Xunstdize_final[nb_samples:nb_points_plot,0],CBO_Xunstdize_final[nb_samples:nb_points_plot,1],color='red',zorder=1e4)
    ax1.scatter(X_current_best[0],X_current_best[1],color='yellow',marker='*',zorder=1e4)
    # ax1.set_zscale('log', base=2)
    # fig.colorbar(s, shrink=0.7, aspect=10)
    # ax1.plot_surface(X_plot, Y_plot, Z_plot, alpha=0.5,zorder=100)
    ax1.set_xlabel('$x_{up}$')
    ax1.set_ylabel('$x_{down}$')
    # ax1.set_xlim(-0.2,-0.198)
    # ax1.set_ylim(-0.2,-0.198)
    # ax1.set_zlim(-8,10)

    # ax1.set_title('Acquisition function (log-scale)')
    fig.savefig(f'optim_2D_sloshing_CEI_{nb_it_plot:02}.pdf')

    #####################################################################################################
    ######################################################################################################
    ######################################################################################################
    ######################################################################################################
    fig = plt.figure()
    ax1=plt.subplot(111,projection='3d')
    s = ax1.plot_surface(X_plot,Y_plot,np.log(dataACQgrid),cmap=plt.cm.viridis)
    # ax1.plot(X_current_best[0],X_current_best[1],color='yellow',marker='*')
    # ax1.set_zscale('log', base=2)
    # fig.colorbar(s, shrink=0.7, aspect=10)
    # ax1.plot_surface(X_plot, Y_plot, Z_plot, alpha=0.5,zorder=100)
    ax1.set_xlabel('$x_{up}$')
    ax1.set_ylabel('$x_{down}$')
    # ax1.set_xlim(-0.2,-0.198)
    # ax1.set_ylim(-0.2,-0.198)
    # ax1.set_zlim(-8,10)

    # ax1.set_title('Acquisition function (log-scale)')
    fig.savefig(f'optim_2D_sloshing_CEI_3D_{nb_it_plot:02}.pdf')

    ######################################################################################################
    ######################################################################################################
    ######################################################################################################
    ######################################################################################################
    fig = plt.figure()
    ax1=plt.subplot(111,projection='3d')
    s = ax1.plot_surface(X_plot,Y_plot,dataGPObjgrid*1e3,cmap=plt.cm.viridis)
    ax1.scatter(CBO_Xunstdize_final[0:nb_points_plot,0],CBO_Xunstdize_final[0:nb_points_plot,1],CBO_Zunstdize_final[0:nb_points_plot].numpy().flatten()*1e3,color='red',s=20)

    # ax1.set_zscale('log', base=2)
    # fig.colorbar(s, shrink=0.7, aspect=10)
    # ax1.plot_surface(X_plot, Y_plot, Z_plot, alpha=0.5,zorder=100)
    ax1.set_xlabel('$x_{up}$')
    ax1.set_ylabel('$x_{down}$')
    # ax1.set_zlim(-8,10)
    ax1.set_title(f'GP obj at iteration {nb_it_plot+1}')
    #####################################################################################################
    ######################################################################################################
    ######################################################################################################
    ######################################################################################################
    fig = plt.figure()
    ax1=plt.subplot(111)
    s = ax1.contour(X_plot,Y_plot,dataGPObjgrid*1e3,cmap=plt.cm.viridis)
    ax1.contour(X_plot, Y_plot, dataGPConsgrid, [0], colors='red')
    plt.contourf(X_plot, Y_plot, dataGPConsgrid, levels=[-10, 0], colors=["red"], alpha=0.2)
    ax1.scatter(CBO_Xunstdize_final[0:nb_points_plot,0],CBO_Xunstdize_final[0:nb_points_plot,1],color='red',s=20)
    ax1.scatter(CBO_Xunstdize_final[0:nb_samples,0],CBO_Xunstdize_final[0:nb_samples,1],color='blue',zorder=1e4)
    ax1.scatter(CBO_Xunstdize_final[nb_samples:nb_points_plot,0],CBO_Xunstdize_final[nb_samples:nb_points_plot,1],color='red',zorder=1e4)
    ax1.scatter(X_current_best[0],X_current_best[1],color='yellow',marker='*',zorder=1e4)
    # ax1.set_zscale('log', base=2)
    # fig.colorbar(s, shrink=0.7, aspect=10)
    # ax1.plot_surface(X_plot, Y_plot, Z_plot, alpha=0.5,zorder=100)
    ax1.set_xlabel('$x_{up}$')
    ax1.set_ylabel('$x_{down}$')
    # ax1.set_zlim(-8,10)
    # ax1.set_title(f'GP obj at iteration {nb_it_plot+1}')
    fig.savefig(f'optim_2D_sloshing_OBJ+cons_{nb_it_plot:02}.pdf')



    #####################################################################################################
    ######################################################################################################
    ######################################################################################################
    ######################################################################################################
    fig = plt.figure()
    ax1=plt.subplot(111,projection='3d')
    s = ax1.plot_surface(X_plot,Y_plot,dataGPConsgrid,cmap=plt.cm.viridis)
    ax1.scatter(CBO_Xunstdize_final[0:nb_points_plot,0],CBO_Xunstdize_final[0:nb_points_plot,1],CBO_Cunstdize_final[0:nb_points_plot].numpy().flatten(),color='red',s=20)
    # ax1.set_zscale('log', base=2)
    # fig.colorbar(s, shrink=0.7, aspect=10)
    # ax1.plot_surface(X_plot, Y_plot, Z_plot, alpha=0.5,zorder=100)
    ax1.set_xlabel('$x_{up}$')
    ax1.set_ylabel('$x_{down}$')
    # ax1.set_zlim(-8,10)
    ax1.set_title(f'GP cons at iteration {nb_it_plot+1}')

In [ ]:
import pymumps

In [ ]:
import torch

from botorch.fit import fit_gpytorch_mll
from botorch.models import SingleTaskGP
from botorch.test_functions import Hartmann
from gpytorch.mlls import ExactMarginalLogLikelihood

neg_hartmann6 = Hartmann(dim=6, negate=True)

torch.manual_seed(seed=12345)  # to keep the data conditions the same
dtype = torch.float64
train_x = torch.rand(10, 6, dtype=dtype)
train_obj = neg_hartmann6(train_x).unsqueeze(-1)
model = SingleTaskGP(train_X=train_x, train_Y=train_obj)
mll = ExactMarginalLogLikelihood(model.likelihood, model)
fit_gpytorch_mll(mll);

from botorch.acquisition.analytic import LogExpectedImprovement

best_value = train_obj.max()
LogEI = LogExpectedImprovement(model=model, best_f=best_value)

from botorch.optim import optimize_acqf

new_point_analytic, _ = optimize_acqf(
    acq_function=LogEI,
    bounds=torch.tensor([[0.0] * 6, [1.0] * 6]),
    q=1,
    num_restarts=20,
    raw_samples=100,
    options={},
)

In [ ]:
model

In [ ]:
model.covar_module.batch_shape